# Clean Stale Drive

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [ ]:
!rm -rf /content/drive


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Clone/Pull Repo

In [ ]:
%cd /content
!rm -rf Dataset-Attribution
!git clone https://github.com/ARK-7-AI/Dataset-Attribution.git Dataset-Attribution
%cd /content/Dataset-Attribution
!ls -la
!test -f pyproject.toml && echo "pyproject found"
!pip install -U pip
!pip install -e .

/content
Cloning into 'Dataset-Attribution'...
remote: Enumerating objects: 735, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 735 (delta 6), reused 0 (delta 0), pack-reused 672 (from 4)
Receiving objects: 100% (735/735), 7.07 MiB | 12.07 MiB/s, done.
Resolving deltas: 100% (453/453), done.
/content/Dataset-Attribution
total 76
drwxr-xr-x 10 root root  4096 Apr 16 01:26 .
drwxr-xr-x  1 root root  4096 Apr 16 01:26 ..
drwxr-xr-x  3 root root  4096 Apr 16 01:26 configs
drwxr-xr-x  3 root root  4096 Apr 16 01:26 data
drwxr-xr-x  2 root root  4096 Apr 16 01:26 docs
drwxr-xr-x  8 root root  4096 Apr 16 01:26 .git
-rw-r--r--  1 root root  4688 Apr 16 01:26 .gitignore
drwxr-xr-x  3 root root  4096 Apr 16 01:26 outputs
-rw-r--r--  1 root root   609 Apr 16 01:26 pyproject.toml
-rw-r--r--  1 root root   115 Apr 16 01:26 pytest.ini
-rw-r--r--  1 root root 19762 Apr 16 01:26 README.md
drwxr-xr-x  2 root root  4096 Apr 16 01:26 sc

In [ ]:
from src.data.split import load_split_config, run_split

cfg = load_split_config("configs/data.yaml")
result = run_split(cfg)

print("Split dir:", result["split_dir"])
print("Counts:", result["counts"])
print("Manifests:", {k: str(v) for k, v in result["manifests"].items()})

Split dir: outputs/runs/default_run/splits
Counts: {'train': 2400, 'test': 300, 'shadow': 300}
Manifests: {'train': 'outputs/runs/default_run/splits/train.csv', 'test': 'outputs/runs/default_run/splits/test.csv', 'shadow': 'outputs/runs/default_run/splits/shadow.csv'}


In [ ]:
!python -m src.training.lora_train --config configs/train_lora.final.yaml

[startup] effective_versions transformers=5.0.0 accelerate=1.13.0 peft=0.18.1
[timing] phase=config_path_validation status=start
[timing] phase=config_path_validation status=end elapsed_s=0.045
[timing] phase=model_tokenizer_load status=start
config.json: 100% 661/661 [00:00<00:00, 7.74MB/s]
tokenizer_config.json: 7.30kB [00:00, 40.3MB/s]
vocab.json: 2.78MB [00:00, 65.5MB/s]
merges.txt: 1.67MB [00:00, 197MB/s]
tokenizer.json: 7.03MB [00:00, 266MB/s]
model.safetensors.index.json: 35.6kB [00:00, 191MB/s]
Fetching 2 files: 100% 2/2 [00:22<00:00, 11.01s/it]
Download complete: 100% 6.17G/6.17G [00:22<00:00, 280MB/s] 
Loading weights: 100% 434/434 [00:00<00:00, 1039.06it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 3.36MB/s]
[timing] phase=model_tokenizer_load status=end elapsed_s=27.045
[timing] phase=dataset_load_tokenization status=start
[data-prep-cache] normalized_hit=False train_hit=False tokenizer_model_id=Qwen/Qwen2.5-3B-Instruct max_se

# **Save Training Artifacts to Drive**
# LoRA Training Pipeline and Artifact Management

This notebook prepares the dataset split, runs LoRA fine-tuning, and saves the resulting artifacts to Google Drive.  
It also provides snapshot inspection and restoration steps for reproducible training and later attribution experiments.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/content/Dataset-Attribution"
DRIVE_ROOT="/content/drive/MyDrive/dataset-attribution-artifacts"

TRAIN_RUN_ID="final_report_run"
SPLIT_RUN_ID="default_run"

cd "${REPO_DIR}"

# Snapshot ID: git short hash + timestamp
GIT_HASH="$(git rev-parse --short HEAD 2>/dev/null || echo nohash)"
TS="$(date -u +%Y%m%dT%H%M%SZ)"
SNAPSHOT_ID="${GIT_HASH}_${TS}"

SNAPSHOT_DIR="${DRIVE_ROOT}/${SNAPSHOT_ID}"
DST_TRAIN="${SNAPSHOT_DIR}/runs/${TRAIN_RUN_ID}/train"
DST_SPLITS="${SNAPSHOT_DIR}/runs/${SPLIT_RUN_ID}/splits"

SRC_TRAIN="outputs/runs/${TRAIN_RUN_ID}/train"
SRC_SPLITS="outputs/runs/${SPLIT_RUN_ID}/splits"

echo "Saving snapshot: ${SNAPSHOT_ID}"
echo "  train  : ${SRC_TRAIN} -> ${DST_TRAIN}"
echo "  splits : ${SRC_SPLITS} -> ${DST_SPLITS}"

# Pre-check source existence
[ -d "${SRC_TRAIN}/adapter" ]   || { echo "Missing ${SRC_TRAIN}/adapter"; exit 1; }
[ -d "${SRC_TRAIN}/tokenizer" ] || { echo "Missing ${SRC_TRAIN}/tokenizer"; exit 1; }
[ -f "${SRC_SPLITS}/train.csv" ] || { echo "Missing ${SRC_SPLITS}/train.csv"; exit 1; }
[ -f "${SRC_SPLITS}/test.csv" ]  || { echo "Missing ${SRC_SPLITS}/test.csv"; exit 1; }

# Validate split overlap before saving
python - <<'PY' "${SRC_SPLITS}/train.csv" "${SRC_SPLITS}/test.csv"
import csv, sys
trp, tep = sys.argv[1], sys.argv[2]
with open(trp, newline="", encoding="utf-8") as f:
    tr = [r["sample_id"].strip() for r in csv.DictReader(f) if (r.get("sample_id") or "").strip()]
with open(tep, newline="", encoding="utf-8") as f:
    te = [r["sample_id"].strip() for r in csv.DictReader(f) if (r.get("sample_id") or "").strip()]
ov = sorted(set(tr) & set(te))
print("train:", len(tr), "test:", len(te), "overlap:", len(ov))
if ov:
    raise SystemExit(f"ERROR overlap sample_id(s): {ov[:10]}")
print("Split validation OK")
PY

# Save atomically into a temp dir then move into place
TMP_DIR="${SNAPSHOT_DIR}.tmp"
rm -rf "${TMP_DIR}"
mkdir -p "${TMP_DIR}/runs/${TRAIN_RUN_ID}" "${TMP_DIR}/runs/${SPLIT_RUN_ID}"

rsync -a --delete "${SRC_TRAIN}/"  "${TMP_DIR}/runs/${TRAIN_RUN_ID}/train/"
rsync -a --delete "${SRC_SPLITS}/" "${TMP_DIR}/runs/${SPLIT_RUN_ID}/splits/"

# Metadata
cat > "${TMP_DIR}/metadata.json" <<JSON
{
  "snapshot_id": "${SNAPSHOT_ID}",
  "created_utc": "${TS}",
  "git_hash": "${GIT_HASH}",
  "train_run_id": "${TRAIN_RUN_ID}",
  "split_run_id": "${SPLIT_RUN_ID}",
  "repo_dir": "${REPO_DIR}"
}
JSON

mv "${TMP_DIR}" "${SNAPSHOT_DIR}"

echo "Saved snapshot at: ${SNAPSHOT_DIR}"
echo "Use this COMMIT_HASH value later: ${SNAPSHOT_ID}"

Saving snapshot: d9121b2_20260416T013135Z
  train  : outputs/runs/final_report_run/train -> /content/drive/MyDrive/dataset-attribution-artifacts/d9121b2_20260416T013135Z/runs/final_report_run/train
  splits : outputs/runs/default_run/splits -> /content/drive/MyDrive/dataset-attribution-artifacts/d9121b2_20260416T013135Z/runs/default_run/splits
train: 2400 test: 300 overlap: 0
Split validation OK
Saved snapshot at: /content/drive/MyDrive/dataset-attribution-artifacts/d9121b2_20260416T013135Z
Use this COMMIT_HASH value later: d9121b2_20260416T013135Z


In [ ]:
%%bash
DRIVE_ROOT="/content/drive/MyDrive/dataset-attribution-artifacts"
COMMIT_HASH="8c61627_20260414T192655Z"

echo "=== Snapshot root ==="
ls -la "${DRIVE_ROOT}/${COMMIT_HASH}/" 2>/dev/null || echo "Snapshot folder missing entirely!"

echo ""
echo "=== Runs folder ==="
ls -la "${DRIVE_ROOT}/${COMMIT_HASH}/runs/" 2>/dev/null || echo "No runs/ folder found!"

echo ""
echo "=== Full tree (depth 4) ==="
find "${DRIVE_ROOT}/${COMMIT_HASH}" -maxdepth 4 -print 2>/dev/null || echo "Cannot read snapshot"

=== Snapshot root ===
Snapshot folder missing entirely!

=== Runs folder ===
No runs/ folder found!

=== Full tree (depth 4) ===
Cannot read snapshot


# Retrieve the Artifacts from Drive

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/content/Dataset-Attribution"
REPO_URL="https://github.com/ARK-7-AI/Dataset-Attribution.git"

TRAIN_RUN_ID="final_report_run"
SPLIT_RUN_ID="default_run"

DRIVE_ROOT="/content/drive/MyDrive/dataset-attribution-artifacts"
COMMIT_HASH="079a2e2_20260414T214147Z"   # set exact snapshot ID for reproducible restore

# Ensure repo exists
if [ ! -d "${REPO_DIR}/.git" ]; then
  git clone "${REPO_URL}" "${REPO_DIR}"
fi
cd "${REPO_DIR}"
mkdir -p outputs/runs

# Resolve snapshot folder when COMMIT_HASH=latest
if [ "${COMMIT_HASH}" = "latest" ]; then
  BASE="${DRIVE_ROOT}"
  [ -d "${BASE}" ] || { echo "Drive artifacts root not found: ${BASE}"; exit 1; }
  COMMIT_HASH="$(find "${BASE}" -mindepth 1 -maxdepth 1 -type d -printf '%T@ %f\n' | sort -n | tail -1 | awk '{print $2}')"
fi

SRC_TRAIN="${DRIVE_ROOT}/${COMMIT_HASH}/runs/${TRAIN_RUN_ID}/train"
SRC_SPLITS="${DRIVE_ROOT}/${COMMIT_HASH}/runs/${SPLIT_RUN_ID}/splits"

DST_TRAIN="outputs/runs/${TRAIN_RUN_ID}/train"
DST_SPLITS="outputs/runs/${SPLIT_RUN_ID}/splits"

echo "Restoring snapshot: ${COMMIT_HASH}"
echo "  train  : ${SRC_TRAIN}"
echo "  splits : ${SRC_SPLITS}"

[ -d "${SRC_TRAIN}" ]  || { echo "Missing source train dir: ${SRC_TRAIN}"; exit 1; }
[ -d "${SRC_SPLITS}" ] || { echo "Missing source splits dir: ${SRC_SPLITS}"; exit 1; }

# Clean target to avoid stale mixed files
rm -rf "${DST_TRAIN}" "${DST_SPLITS}"
mkdir -p "${DST_TRAIN}" "${DST_SPLITS}"

# Restore
rsync -a --delete "${SRC_TRAIN}/" "${DST_TRAIN}/"
rsync -a --delete "${SRC_SPLITS}/" "${DST_SPLITS}/"

echo "== Verify required artifacts =="
[ -d "${DST_TRAIN}/adapter" ]   || { echo "Missing ${DST_TRAIN}/adapter"; exit 1; }
[ -d "${DST_TRAIN}/tokenizer" ] || { echo "Missing ${DST_TRAIN}/tokenizer"; exit 1; }
[ -f "${DST_SPLITS}/train.csv" ] || { echo "Missing ${DST_SPLITS}/train.csv"; exit 1; }
[ -f "${DST_SPLITS}/test.csv" ]  || { echo "Missing ${DST_SPLITS}/test.csv"; exit 1; }

python - <<'PY' "${DST_SPLITS}"
import csv, sys
from pathlib import Path

split_dir = Path(sys.argv[1])
trp = split_dir / "train.csv"
tep = split_dir / "test.csv"

with trp.open(newline="", encoding="utf-8") as f:
    tr = [r["sample_id"].strip() for r in csv.DictReader(f) if (r.get("sample_id") or "").strip()]
with tep.open(newline="", encoding="utf-8") as f:
    te = [r["sample_id"].strip() for r in csv.DictReader(f) if (r.get("sample_id") or "").strip()]

ov = sorted(set(tr) & set(te))
print("train:", len(tr), "test:", len(te), "overlap:", len(ov))
if ov:
    raise SystemExit(f"ERROR overlap sample_id(s): {ov[:10]}")
print("Restore validation OK")
PY

echo "Restore complete."

Restoring snapshot: 079a2e2_20260414T214147Z
  train  : /content/drive/MyDrive/dataset-attribution-artifacts/079a2e2_20260414T214147Z/runs/final_report_run/train
  splits : /content/drive/MyDrive/dataset-attribution-artifacts/079a2e2_20260414T214147Z/runs/default_run/splits
== Verify required artifacts ==
train: 2400 test: 300 overlap: 0
Restore validation OK
Restore complete.
